In [1]:
import sys

sys.path.append("src")

# Now you can import your strategy modules
from strategies.my_nse_strategy import MyNSEStrategy, MyNSEStrategyConfig
from nautilus_trader.model import InstrumentId

# Test your strategy configuration
config = MyNSEStrategyConfig(
    instrument_id=InstrumentId.from_str("BANKNIFTY.OPT.26Jun2025.64000.CALL.NSE"),
    order_id_tag="001",
    oms_type="NETTING",
)

print("Strategy config created successfully!")
print(f"Instrument: {config.instrument_id}")
print(f"Position size: {config.position_size}")

Strategy config created successfully!
Instrument: BANKNIFTY.OPT.26Jun2025.64000.CALL.NSE
Position size: 1


In [2]:
# Create a strategy instance
strategy = MyNSEStrategy(config)
print("✅ Strategy instance created successfully!")
print(f"Strategy ID: {strategy.id}")
print(f"Strategy config: {strategy.config}")

✅ Strategy instance created successfully!
Strategy ID: MyNSEStrategy-001
Strategy config: MyNSEStrategyConfig(instrument_id=InstrumentId('BANKNIFTY.OPT.26Jun2025.64000.CALL.NSE'), meta_catalog_path='catalog-meta', sl_pct=0.02, tp_pct=0.03, atm_window='15min', position_size=1, end_time='15:15', min_iv=0, entry_buffer_pct=0.01, lookback_intervals=2, min_oi_change=-100, breakeven_trigger_pct=2, sar_enabled=True, strategy_id=None, order_id_tag='001', use_uuid_client_order_ids=False, oms_type='NETTING', external_order_claims=None, manage_contingent_orders=False, manage_gtd_expiry=False, log_events=True, log_commands=True)


In [3]:
import pandas as pd

# Load your data
data_file = "data/nse/Mock_15-Min_Intraday_Option_Chain.csv"
df = pd.read_csv(data_file)

print(f"✅ Data loaded successfully!")
print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Sample data:")
print(df.head())

✅ Data loaded successfully!
Data shape: (1830, 18)
Columns: ['timestamp', 'symbol', 'option_type', 'strike', 'expiry', 'bid', 'ask', 'last', 'volume', 'open_interest', 'impliedVolatility', 'pchangeinOpenInterest', 'totalBuyQuantity', 'totalSellQuantity', 'underlyingValue', 'underlying', 'identifier', 'pChange']
Date range: 6/18/25 12:00 to 6/18/25 14:15
Sample data:
       timestamp                                  symbol option_type  strike  \
0  6/18/25 12:00  BANKNIFTY.NSE.OPT.26Jun2025.40500.CALL        CALL   40500   
1  6/18/25 12:00  BANKNIFTY.NSE.OPT.26Jun2025.42000.CALL        CALL   42000   
2  6/18/25 12:00  BANKNIFTY.NSE.OPT.26Jun2025.42500.CALL        CALL   42500   
3  6/18/25 12:00  BANKNIFTY.NSE.OPT.26Jun2025.43000.CALL        CALL   43000   
4  6/18/25 12:00  BANKNIFTY.NSE.OPT.26Jun2025.43500.CALL        CALL   43500   

      expiry           bid           ask          last  volume  open_interest  \
0  26-Jun-25  15198.151129  15620.366772  15149.698280      33       

In [4]:
# Import backtest components
from nautilus_trader.backtest.node import (
    BacktestNode,
    BacktestRunConfig,
    BacktestDataConfig,
    BacktestVenueConfig,
)
from nautilus_trader.backtest.engine import BacktestEngineConfig
from nautilus_trader.config import LoggingConfig, ImportableStrategyConfig
from nautilus_trader.model import InstrumentId, QuoteTick
from nautilus_trader.persistence.catalog import ParquetDataCatalog

# Set up backtest configuration
catalog = ParquetDataCatalog("./catalog")
INSTRUMENT_ID = InstrumentId.from_str("BANKNIFTY.OPT.26Jun2025.64000.CALL.NSE")

venue = BacktestVenueConfig(
    name="NSE",
    oms_type="NETTING",
    account_type="MARGIN",
    base_currency="INR",
    starting_balances=["1_000_000 INR"],
)

data = BacktestDataConfig(
    catalog_path="./catalog",
    data_cls=QuoteTick,
    instrument_id=INSTRUMENT_ID,
    start_time="2025-06-18T12:00:00",
    end_time="2025-06-18T14:15:00",
)

strategy_config = ImportableStrategyConfig(
    strategy_path="strategies.my_nse_strategy:MyNSEStrategy",
    config_path="strategies.my_nse_strategy:MyNSEStrategyConfig",
    config=dict(
        instrument_id=INSTRUMENT_ID,
        order_id_tag="001",
        oms_type="NETTING",
    ),
)

engine = BacktestEngineConfig(
    strategies=[strategy_config],
    logging=LoggingConfig(
        log_level="INFO",
        log_level_file="INFO",
    ),
)

config = BacktestRunConfig(
    engine=engine,
    venues=[venue],
    data=[data],
)

# Run the backtest
node = BacktestNode(configs=[config])
results = node.run()

print("✅ Backtest completed!")
if results:
    result = results[0]
    print(f"Total orders: {result.total_orders}")
    print(f"Total positions: {result.total_positions}")
    print(f"Total events: {result.total_events}")

2025-06-21T13:01:57.445863000Z [INFO] BACKTESTER-001.BacktestEngine: =================================================================
2025-06-21T13:01:57.445872000Z [INFO] BACKTESTER-001.BacktestEngine:  NAUTILUS TRADER - Automated Algorithmic Trading Platform
2025-06-21T13:01:57.445872001Z [INFO] BACKTESTER-001.BacktestEngine:  by Nautech Systems Pty Ltd.
2025-06-21T13:01:57.445875000Z [INFO] BACKTESTER-001.BacktestEngine:  Copyright (C) 2015-2025. All rights reserved.
2025-06-21T13:01:57.445875001Z [INFO] BACKTESTER-001.BacktestEngine: =================================================================
2025-06-21T13:01:57.445875002Z [INFO] BACKTESTER-001.BacktestEngine: 
2025-06-21T13:01:57.445875003Z [INFO] BACKTESTER-001.BacktestEngine: ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⣴⣶⡟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
2025-06-21T13:01:57.445876000Z [INFO] BACKTESTER-001.BacktestEngine: ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣰⣾⣿⣿⣿⠀⢸⣿⣿⣿⣿⣶⣶⣤⣀⠀⠀⠀⠀⠀
2025-06-21T13:01:57.445876001Z [INFO] BACKTESTER-001.BacktestEngine: ⠀⠀⠀⠀⠀⠀⢀⣴⡇⢀⣾⣿⣿⣿⣿⣿⠀⣾⣿⣿⣿⣿⣿⣿⣿⠿⠓⠀⠀⠀⠀
2025-06-21T13:01

In [5]:
# Check strategy's internal trade log
if hasattr(strategy, "trades") and strategy.trades:
    print("📊 Strategy Trade Analysis:")
    for i, trade in enumerate(strategy.trades):
        print(f"Trade {i + 1}:")
        print(f"  Entry: {trade['entry_price']} at {trade['entry_time']}")
        print(f"  Exit: {trade['exit_price']} at {trade['exit_time']}")
        print(f"  Reason: {trade['exit_reason']}")
        print(f"  P&L: {trade['exit_price'] - trade['entry_price']:.2f}")
        print()
else:
    print("No trades recorded in strategy")

No trades recorded in strategy


In [7]:
# Display all strategy configuration parameters
print("🔧 Strategy Configuration:")
print(f"  Instrument: {config.instrument_id}")
print(f"  Position Size: {config.position_size}")
print(f"  Take Profit %: {config.tp_pct}%")
print(f"  Entry Buffer %: {config.entry_buffer_pct}%")
print(f"  Lookback Intervals: {config.lookback_intervals}")
print(f"  Min IV: {config.min_iv}")
print(f"  Min OI Change: {config.min_oi_change}")
print(f"  Breakeven Trigger %: {config.breakeven_trigger_pct}%")
print(f"  SAR Enabled: {config.sar_enabled}")
print(f"  End Time: {config.end_time}")

🔧 Strategy Configuration:


AttributeError: 'BacktestRunConfig' object has no attribute 'instrument_id'